# 04 — Metadata, Embeddings & Vector Database
## Person 4 · Phase 1 (preparation for the Phase 2 retrieval system)

Works **directly on the shared Google Drive folder** (same connection method as Person 3). No images are displayed and no data is downloaded; the only download is the CLIP model weights.

| Task | Section |
|---|---|
| 1-2. Extract metadata, organise into structured files | 3-4 |
| 3. Choose and configure embedding model (CLIP) | 5 |
| 4. Image embeddings (knowledge base) | 6 |
| 5. Text embeddings (captions) | 7 |
| 6. Save embeddings + IDs | 8 |
| 7. FAISS index | 9 |
| 8. Retrieval test | 11 |
| 9. Document model/embedding configuration | 12 |
| Quality control + leakage audit | 10 |

In [1]:
# 1. Environment setup (same method as Person 3)
!pip install -q faiss-cpu transformers

import os, json, re, time, shutil
import numpy as np
import pandas as pd
import torch
import faiss
from collections import defaultdict, Counter
from PIL import Image, ImageOps
from torch.utils.data import Dataset, DataLoader
from tqdm.notebook import tqdm
from google.colab import drive

drive.mount('/content/drive')
print("Environment initialized successfully.")

Mounted at /content/drive
Environment initialized successfully.


In [2]:
# 2. Shared project paths
# If the folder is "Shared with me" and not under My Drive, right-click it in Drive ->
# Organize -> Add shortcut -> My Drive, so the path below works.
BASE_DIR = "/content/drive/MyDrive/COCO_Project"

ANNOTATIONS_DIR = os.path.join(BASE_DIR, "annotations")
IMAGES_DIR      = os.path.join(BASE_DIR, "images")
METADATA_DIR    = os.path.join(BASE_DIR, "metadata")
STATISTICS_DIR  = os.path.join(BASE_DIR, "statistics")
TEST_IMAGES_DIR = os.path.join(BASE_DIR, "test", "val2017")

# Person 4 output folders (created inside the shared project)
EMBEDDINGS_DIR = os.path.join(BASE_DIR, "embeddings")
INDEX_DIR      = os.path.join(BASE_DIR, "faiss_index")
P4_META_DIR    = os.path.join(BASE_DIR, "person4_metadata")
for d in [EMBEDDINGS_DIR, INDEX_DIR, P4_META_DIR]:
    os.makedirs(d, exist_ok=True)

for folder in [BASE_DIR, ANNOTATIONS_DIR, IMAGES_DIR, METADATA_DIR]:
    assert os.path.isdir(folder), f"Folder not found: {folder}"

print("Shared project connected:", BASE_DIR)
print("Metadata files available:", len(os.listdir(METADATA_DIR)))

Shared project connected: /content/drive/MyDrive/COCO_Project
Metadata files available: 15


## 3. Load shared files from teammates

In [3]:
# Person 1 / Person 3 outputs
selected_df = pd.read_csv(os.path.join(METADATA_DIR, "selected_image_ids.csv"))
test_df     = pd.read_csv(os.path.join(METADATA_DIR, "person3_test100_manifest.csv"))

kb_df  = selected_df[selected_df["split"] == "knowledge_base"].reset_index(drop=True)
val_df = selected_df[selected_df["split"] == "validation"].reset_index(drop=True)

print("Knowledge base:", len(kb_df), "| Validation:", len(val_df), "| Test:", len(test_df))
assert len(kb_df) == 10000 and len(val_df) == 100 and len(test_df) == 100

# COCO annotations (read directly from the shared Drive)
with open(os.path.join(ANNOTATIONS_DIR, "instances_train2017.json")) as f:
    instances = json.load(f)
with open(os.path.join(ANNOTATIONS_DIR, "captions_train2017.json")) as f:
    captions = json.load(f)
with open(os.path.join(ANNOTATIONS_DIR, "captions_test100.json")) as f:
    captions_test = json.load(f)
print("Annotations loaded.")

Knowledge base: 10000 | Validation: 100 | Test: 100
Annotations loaded.


## 4. Metadata extraction and structured files
Per image: ID, dimensions, object categories, bounding boxes (COCO format `[x, y, width, height]`), captions.
Raw captions are kept unchanged for evaluation; cleaned captions are stored in a **separate field** and used for indexing only.

In [4]:
cat_names = {c["id"]: c["name"] for c in instances["categories"]}
split_of  = dict(zip(selected_df["image_id"], selected_df["split"]))
sel_ids   = set(split_of)

img_info = {im["id"]: im for im in instances["images"] if im["id"] in sel_ids}

objs_by_img = defaultdict(list)
for a in instances["annotations"]:
    if a["image_id"] in sel_ids:
        objs_by_img[a["image_id"]].append({
            "annotation_id": a["id"],
            "category_id": a["category_id"],
            "category": cat_names[a["category_id"]],
            "bbox": [round(float(v), 2) for v in a["bbox"]],   # [x, y, w, h]
            "area": round(float(a["area"]), 2),
            "iscrowd": int(a["iscrowd"]),
        })

caps_by_img = defaultdict(list)
for a in captions["annotations"]:
    if a["image_id"] in sel_ids:
        caps_by_img[a["image_id"]].append({"caption_id": a["id"], "caption_raw": a["caption"]})

# Caption cleaning: for retrieval / indexing ONLY
def clean_caption(t):
    t = t.lower().strip()
    t = re.sub(r"[^a-z0-9\s']", " ", t)
    return re.sub(r"\s+", " ", t).strip()

for lst in caps_by_img.values():
    for c in lst:
        c["caption_clean"] = clean_caption(c["caption_raw"])

# Assemble the metadata records
records = []
for image_id in selected_df["image_id"]:
    im = img_info[image_id]
    objs = objs_by_img.get(image_id, [])
    records.append({
        "image_id": int(image_id),
        "split": split_of[image_id],
        "file_name": im["file_name"],
        "width": im["width"], "height": im["height"],
        "num_objects": len(objs),
        "object_categories": sorted({o["category"] for o in objs}),
        "objects": objs,
        "captions": caps_by_img.get(image_id, []),
    })
print("Metadata records:", len(records))
print("Images without objects:", sum(r["num_objects"] == 0 for r in records))
print("Images without captions:", sum(len(r["captions"]) == 0 for r in records))

Metadata records: 10100
Images without objects: 104
Images without captions: 0


In [5]:
# Save structured files
# (a) full JSON (nested: objects + bounding boxes + captions)
with open(os.path.join(P4_META_DIR, "person4_image_metadata.json"), "w") as f:
    json.dump(records, f)

# (b) image-level CSV
pd.DataFrame([{
    "image_id": r["image_id"], "split": r["split"], "file_name": r["file_name"],
    "width": r["width"], "height": r["height"], "num_objects": r["num_objects"],
    "object_categories": "|".join(r["object_categories"]),
    "num_captions": len(r["captions"]),
} for r in records]).to_csv(os.path.join(P4_META_DIR, "person4_image_metadata.csv"), index=False)

# (c) object / bounding-box CSV
pd.DataFrame([{"image_id": r["image_id"], "split": r["split"], **o}
              for r in records for o in r["objects"]]
            ).to_csv(os.path.join(P4_META_DIR, "person4_object_annotations.csv"), index=False)

# (d) caption CSV (raw + cleaned stored in separate columns)
caption_rows = [{"caption_id": c["caption_id"], "image_id": r["image_id"], "split": r["split"],
                 "caption_raw": c["caption_raw"], "caption_clean": c["caption_clean"]}
                for r in records for c in r["captions"]]
captions_all_df = pd.DataFrame(caption_rows)
captions_all_df.to_csv(os.path.join(P4_META_DIR, "person4_captions.csv"), index=False)

# (e) test-set metadata (captions from Person 3's captions_test100.json, kept raw)
test_caps = defaultdict(list)
for a in captions_test["annotations"]:
    test_caps[a["image_id"]].append({"caption_id": a["id"], "caption_raw": a["caption"]})
test_records = [{"image_id": int(r.image_id), "split": "test", "file_name": r.file_name,
                 "width": int(r.width), "height": int(r.height),
                 "captions": test_caps[int(r.image_id)]} for r in test_df.itertuples()]
with open(os.path.join(P4_META_DIR, "person4_test_metadata.json"), "w") as f:
    json.dump(test_records, f)

print("Saved metadata files in:", P4_META_DIR)
print("Total captions (KB+val):", len(captions_all_df))

Saved metadata files in: /content/drive/MyDrive/COCO_Project/person4_metadata
Total captions (KB+val): 50521


## 5. Embedding model configuration
CLIP `openai/clip-vit-base-patch32` (same model Person 3 used for preprocessing): image + text encoders share one 512-d space, which supports image→image, image→text and text→image retrieval. Embeddings are L2-normalised, so inner product = cosine similarity.

In [6]:
from transformers import CLIPModel, CLIPProcessor

MODEL_NAME = "openai/clip-vit-base-patch32"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
IMG_BATCH, TXT_BATCH = 64, 256

model = CLIPModel.from_pretrained(MODEL_NAME).to(DEVICE).eval()
processor = CLIPProcessor.from_pretrained(MODEL_NAME)
EMB_DIM = model.config.projection_dim

print("Model:", MODEL_NAME, "| Device:", DEVICE, "| Embedding dim:", EMB_DIM)
if DEVICE == "cpu":
    print("WARNING: no GPU. Enable Runtime > Change runtime type > GPU for faster embedding.")

def l2_normalize(x):
    return x / np.clip(np.linalg.norm(x, axis=1, keepdims=True), 1e-12, None)

def _as_tensor(out):
    return out if torch.is_tensor(out) else out.pooler_output

config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  605MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  605MB            

model.safetensors: downloading bytes:           |  0.00B            

... (3 outputs left out)

merges.txt:   0%|          | 0.00/525k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.22M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

Model: openai/clip-vit-base-patch32 | Device: cuda | Embedding dim: 512


## 6. Image embeddings (read directly from Drive, EXIF-corrected, RGB)

In [7]:
class ImageDataset(Dataset):
    def __init__(self, paths): self.paths = paths
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        try:
            img = ImageOps.exif_transpose(Image.open(self.paths[i])).convert("RGB")
            pv = processor.image_processor(images=img, return_tensors="pt")["pixel_values"][0]
            return pv, 1
        except Exception:
            return torch.zeros(3, 224, 224), 0

@torch.no_grad()
def embed_images(paths, ckpt_name=None, ckpt_every=20):
    ckpt = os.path.join(EMBEDDINGS_DIR, f"_ckpt_{ckpt_name}.npz") if ckpt_name else None
    feats, flags = [], []
    if ckpt and os.path.exists(ckpt):                       # resume after a disconnect
        d = np.load(ckpt); feats, flags = [d["f"]], [d["ok"]]
        print("Resuming from", sum(len(x) for x in flags), "images")
    start = sum(len(x) for x in flags)
    ds = torch.utils.data.Subset(ImageDataset(paths), range(start, len(paths)))
    dl = DataLoader(ds, batch_size=IMG_BATCH, num_workers=2)
    for b, (pv, ok) in enumerate(tqdm(dl, desc="Image embeddings")):
        out = _as_tensor(model.get_image_features(pixel_values=pv.to(DEVICE)))
        feats.append(out.float().cpu().numpy()); flags.append(ok.numpy())
        if ckpt and b % ckpt_every == 0:
            np.savez(ckpt, f=np.concatenate(feats), ok=np.concatenate(flags))
    F, OK = np.concatenate(feats), np.concatenate(flags)
    return l2_normalize(F).astype("float32"), OK

kb_paths = [os.path.join(IMAGES_DIR, "knowledge_base", fn) for fn in kb_df["file_name"]]
t0 = time.time()
img_emb_kb, ok_kb = embed_images(kb_paths, ckpt_name="kb")
t_img_kb = time.time() - t0
print("KB image embeddings:", img_emb_kb.shape, "| failed reads:", int((ok_kb == 0).sum()),
      "| time: %.1fs" % t_img_kb)

Image embeddings:   0%|          | 0/157 [00:00<?, ?it/s]

KB image embeddings: (10000, 512) | failed reads: 0 | time: 2102.9s


In [8]:
# Query-side image embeddings for later phases (validation + test); never added to the KB index
val_paths  = [os.path.join(IMAGES_DIR, "validation", fn) for fn in val_df["file_name"]]
test_paths = [os.path.join(TEST_IMAGES_DIR, fn) for fn in test_df["file_name"]]
img_emb_val,  ok_val  = embed_images(val_paths)
img_emb_test, ok_test = embed_images(test_paths)
print("Validation:", img_emb_val.shape, "failed:", int((ok_val == 0).sum()))
print("Test:", img_emb_test.shape, "failed:", int((ok_test == 0).sum()))

Image embeddings:   0%|          | 0/2 [00:00<?, ?it/s]

Image embeddings:   0%|          | 0/2 [00:00<?, ?it/s]

Validation: (100, 512) failed: 0
Test: (100, 512) failed: 0


## 7. Text embeddings for captions (knowledge-base captions only, cleaned text)

In [9]:
kb_caps_df = captions_all_df[captions_all_df["split"] == "knowledge_base"].reset_index(drop=True)
print("KB captions:", len(kb_caps_df))

@torch.no_grad()
def embed_texts(texts, batch=TXT_BATCH):
    out = []
    for i in tqdm(range(0, len(texts), batch), desc="Text embeddings"):
        tok = processor.tokenizer(texts[i:i+batch], padding=True, truncation=True,
                                  max_length=77, return_tensors="pt").to(DEVICE)
        out.append(_as_tensor(model.get_text_features(**tok)).float().cpu().numpy())
    return l2_normalize(np.concatenate(out)).astype("float32")

t0 = time.time()
txt_emb_kb = embed_texts(kb_caps_df["caption_clean"].tolist())
t_txt_kb = time.time() - t0
print("KB text embeddings:", txt_emb_kb.shape, "| time: %.1fs" % t_txt_kb)

# Validation caption embeddings (for tuning phase queries)
val_caps_df = captions_all_df[captions_all_df["split"] == "validation"].reset_index(drop=True)
txt_emb_val = embed_texts(val_caps_df["caption_clean"].tolist())

KB captions: 50021


Text embeddings:   0%|          | 0/196 [00:00<?, ?it/s]

KB text embeddings: (50021, 512) | time: 49.6s


Text embeddings:   0%|          | 0/2 [00:00<?, ?it/s]

## 8. Save embeddings with their image / caption IDs (`.npy`)

In [10]:
def save(name, arr): np.save(os.path.join(EMBEDDINGS_DIR, name), arr)

save("image_embeddings_kb.npy",       img_emb_kb)
save("image_ids_kb.npy",              kb_df["image_id"].to_numpy(dtype="int64"))
save("text_embeddings_kb.npy",        txt_emb_kb)
save("caption_ids_kb.npy",            kb_caps_df["caption_id"].to_numpy(dtype="int64"))
save("caption_image_ids_kb.npy",      kb_caps_df["image_id"].to_numpy(dtype="int64"))

save("image_embeddings_validation.npy", img_emb_val)
save("image_ids_validation.npy",        val_df["image_id"].to_numpy(dtype="int64"))
save("text_embeddings_validation.npy",  txt_emb_val)
save("caption_ids_validation.npy",      val_caps_df["caption_id"].to_numpy(dtype="int64"))
save("image_embeddings_test.npy",       img_emb_test)
save("image_ids_test.npy",              test_df["image_id"].to_numpy(dtype="int64"))

# remove checkpoint files
for fn in os.listdir(EMBEDDINGS_DIR):
    if fn.startswith("_ckpt_"): os.remove(os.path.join(EMBEDDINGS_DIR, fn))
print(sorted(os.listdir(EMBEDDINGS_DIR)))

['caption_ids_kb.npy', 'caption_ids_validation.npy', 'caption_image_ids_kb.npy', 'image_embeddings_kb.npy', 'image_embeddings_test.npy', 'image_embeddings_validation.npy', 'image_ids_kb.npy', 'image_ids_test.npy', 'image_ids_validation.npy', 'text_embeddings_kb.npy', 'text_embeddings_validation.npy']


## 9. FAISS indexes
`IndexFlatIP` (exact search) on L2-normalised vectors = cosine similarity. 10k images / 50k captions is small enough for exact search. Wrapped in `IndexIDMap2` so results return COCO image IDs / caption IDs directly.

In [11]:
def build_index(vectors, ids):
    t0 = time.time()
    index = faiss.IndexIDMap2(faiss.IndexFlatIP(vectors.shape[1]))
    index.add_with_ids(vectors, ids.astype("int64"))
    return index, time.time() - t0

image_ids_kb   = kb_df["image_id"].to_numpy(dtype="int64")
caption_ids_kb = kb_caps_df["caption_id"].to_numpy(dtype="int64")

image_index, t_idx_img = build_index(img_emb_kb, image_ids_kb)
text_index,  t_idx_txt = build_index(txt_emb_kb, caption_ids_kb)

faiss.write_index(image_index, os.path.join(INDEX_DIR, "image_index_kb.faiss"))
faiss.write_index(text_index,  os.path.join(INDEX_DIR, "text_index_kb.faiss"))

# Link every vector to its image ID, captions and metadata
meta_lookup = {
    str(r["image_id"]): {"file_name": r["file_name"], "width": r["width"], "height": r["height"],
                         "object_categories": r["object_categories"],
                         "caption_ids": [c["caption_id"] for c in r["captions"]]}
    for r in records if r["split"] == "knowledge_base"}
with open(os.path.join(INDEX_DIR, "image_id_to_metadata.json"), "w") as f:
    json.dump(meta_lookup, f)
caption_lookup = kb_caps_df.set_index("caption_id")[["image_id", "caption_raw", "caption_clean"]]
caption_lookup.to_csv(os.path.join(INDEX_DIR, "caption_id_to_text.csv"))

print("Image index vectors:", image_index.ntotal, "| build %.4fs" % t_idx_img)
print("Text index vectors: ", text_index.ntotal,  "| build %.4fs" % t_idx_txt)

Image index vectors: 10000 | build 0.0511s
Text index vectors:  50021 | build 0.0844s


## 10. Quality control and leakage audit

In [12]:
kb_img_set  = set(image_ids_kb.tolist())
val_img_set = set(val_df["image_id"].tolist())
test_img_set = set(test_df["image_id"].tolist())

qc = {
 "image_embeddings_nan":      int(np.isnan(img_emb_kb).sum()),
 "text_embeddings_nan":       int(np.isnan(txt_emb_kb).sum()),
 "unreadable_kb_images":      int((ok_kb == 0).sum()),
 "zero_vectors_image":        int((np.linalg.norm(img_emb_kb, axis=1) < 0.5).sum()),
 "duplicate_image_ids":       int(len(image_ids_kb) - len(set(image_ids_kb.tolist()))),
 "duplicate_caption_ids":     int(len(caption_ids_kb) - len(set(caption_ids_kb.tolist()))),
 "images_without_captions":   int(len(kb_img_set - set(kb_caps_df["image_id"]))),
 "captions_with_unknown_image": int(len(set(kb_caps_df["image_id"]) - kb_img_set)),
 "image_vs_metadata_count_mismatch": int(len(img_emb_kb) != len(kb_df)),
 "caption_vs_embedding_mismatch":    int(len(txt_emb_kb) != len(kb_caps_df)),
 "KB_and_validation_overlap": len(kb_img_set & val_img_set),
 "KB_and_test_overlap":       len(kb_img_set & test_img_set),
 "validation_and_test_overlap": len(val_img_set & test_img_set),
 "test_caption_ids_in_KB_index": len(set(a["id"] for a in captions_test["annotations"]) & set(caption_ids_kb.tolist())),
 "test_images_in_KB_index":   len(set(image_ids_kb.tolist()) & test_img_set),
}
for k, v in qc.items(): print(f"{'OK ' if v == 0 else 'FAIL'}  {k}: {v}")
print("\nQC:", "PASSED" if all(v == 0 for v in qc.values()) else "FAILED")

OK   image_embeddings_nan: 0
OK   text_embeddings_nan: 0
OK   unreadable_kb_images: 0
OK   zero_vectors_image: 0
OK   duplicate_image_ids: 0
OK   duplicate_caption_ids: 0
OK   images_without_captions: 0
OK   captions_with_unknown_image: 0
OK   image_vs_metadata_count_mismatch: 0
OK   caption_vs_embedding_mismatch: 0
OK   KB_and_validation_overlap: 0
OK   KB_and_test_overlap: 0
OK   validation_and_test_overlap: 0
OK   test_caption_ids_in_KB_index: 0
OK   test_images_in_KB_index: 0

QC: PASSED


## 11. Retrieval tests (printed as IDs and text; no images displayed)

In [13]:
K = 5
# Test A: self-retrieval (KB image queried against its own index must return itself at rank 1)
D, I = image_index.search(img_emb_kb[:200], 1)
self_acc = float((I[:, 0] == image_ids_kb[:200]).mean())
print("A) Self-retrieval top-1 accuracy (200 KB images):", self_acc)

# Test B: image -> image. Validation images queried against the KB.
#    Proxy for relevance: overlap of COCO object categories (Jaccard) vs random KB images.
cats = {r["image_id"]: set(r["object_categories"]) for r in records}
def jacc(a, b): return len(a & b) / len(a | b) if (a | b) else 0.0

D, I = image_index.search(img_emb_val, K)
ret_j = np.mean([jacc(cats[q], cats[i]) for q, row in zip(val_df["image_id"], I) for i in row])
rng = np.random.default_rng(42)
rand_j = np.mean([jacc(cats[q], cats[int(rng.choice(image_ids_kb))])
                  for q in val_df["image_id"] for _ in range(K)])
print(f"B) Image->image mean category Jaccard @ {K}: retrieved={ret_j:.3f} vs random={rand_j:.3f}")

# Test C: image -> text. Show reference vs retrieved captions for 3 validation images
D, I = text_index.search(img_emb_val[:3], K)
for n in range(3):
    qid = int(val_df["image_id"][n])
    print(f"\nC) Validation image {qid}")
    print("   Own captions   :", [c["caption_raw"] for c in records[len(kb_df)+n]["captions"][:2]])
    for cid, s in zip(I[n][:3], D[n][:3]):
        print(f"   retrieved ({s:.3f}):", caption_lookup.loc[cid, "caption_raw"])

# Test D: text -> image (validation caption queries the KB image index)
D, I = image_index.search(txt_emb_val[:3], 3)
for n in range(3):
    print(f"\nD) Query: {val_caps_df['caption_raw'][n]}")
    print("   Top image IDs:", I[n].tolist(), "| categories:", [sorted(cats[int(i)])[:4] for i in I[n]])

# Test E: retrieval latency
t0 = time.time()
for q in img_emb_val: image_index.search(q[None, :], 10)
latency_ms = (time.time() - t0) / len(img_emb_val) * 1000
print("\nE) Mean retrieval latency (top-10): %.3f ms/query" % latency_ms)

# Test F: anti-leakage helper — a query image is never retrieved for itself
def retrieve_images(query_vec, query_image_id=None, k=10):
    D, I = image_index.search(query_vec[None, :], k + 1)
    pairs = [(int(i), float(d)) for i, d in zip(I[0], D[0]) if int(i) != query_image_id]
    return pairs[:k]
print("F) Self-exclusion works:", all(i != int(image_ids_kb[0]) for i, _ in retrieve_images(img_emb_kb[0], int(image_ids_kb[0]))))

A) Self-retrieval top-1 accuracy (200 KB images): 1.0
B) Image->image mean category Jaccard @ 5: retrieved=0.475 vs random=0.086

C) Validation image 35030
   Own captions   : ['a person getting ready to hit a serve in tennis', 'A tennis player about to hit plays on a lavender tennis court.']
   retrieved (0.369): A woman in purple shirt and white dress with sneakers and tennis racket.
   retrieved (0.363): A woman on a court with a tennis racket in the air.
   retrieved (0.361): Woman in a white tennis outfit standing on a court with a racket. 

C) Validation image 332039
   Own captions   : ['Chocolate dipped strawberries sitting next to to a banana.', '3 fruit candies next to a banana ']
   retrieved (0.305): A cup of chocolate is sitting next to bowls of fruit.
   retrieved (0.305): A chocolate  covered treat with white chocolate hardened on the top
   retrieved (0.302): Fruits, cheese, chocolate, and other foods cover a table.

C) Validation image 434787
   Own captions   : ['A bo

## 12. Document model and embedding configuration

In [14]:
config = {
  "responsible_person": "Person 4",
  "embedding_model": {
    "name": MODEL_NAME, "type": "CLIP ViT-B/32 (image + text encoders)",
    "embedding_dimension": int(EMB_DIM), "normalisation": "L2",
    "image_batch_size": IMG_BATCH, "text_batch_size": TXT_BATCH,
    "text_max_tokens": 77, "device": DEVICE,
    "library_versions": {"torch": torch.__version__,
                         "transformers": __import__("transformers").__version__,
                         "faiss": faiss.__version__},
  },
  "image_preprocessing": "EXIF orientation fix, RGB conversion, CLIPImageProcessor (resize+center-crop 224, normalise)",
  "text_preprocessing": "lowercase, strip punctuation, collapse whitespace (indexing only; raw captions preserved)",
  "vector_database": {
    "library": "FAISS", "index_type": "IndexIDMap2(IndexFlatIP) — exact search",
    "similarity_metric": "inner product on L2-normalised vectors (= cosine similarity)",
    "image_index_size": int(image_index.ntotal), "text_index_size": int(text_index.ntotal),
    "image_index_file_mb": round(os.path.getsize(os.path.join(INDEX_DIR, "image_index_kb.faiss"))/1e6, 2),
    "text_index_file_mb": round(os.path.getsize(os.path.join(INDEX_DIR, "text_index_kb.faiss"))/1e6, 2),
    "image_index_build_seconds": round(t_idx_img, 4),
    "text_index_build_seconds": round(t_idx_txt, 4),
    "image_embedding_seconds": round(t_img_kb, 1),
    "text_embedding_seconds": round(t_txt_kb, 1),
    "mean_query_latency_ms": round(latency_ms, 3),
  },
  "data": {
    "knowledge_base_images": len(kb_df), "knowledge_base_captions": len(kb_caps_df),
    "validation_images": len(val_df), "test_images": len(test_df),
    "index_contains_only": "knowledge_base images/captions",
  },
  "quality_control": qc,
  "retrieval_test": {"self_retrieval_top1_accuracy_200": self_acc,
                     "image_to_image_category_jaccard_at5": round(float(ret_j), 4),
                     "random_baseline_jaccard": round(float(rand_j), 4)},
  "files": {
    "embeddings": sorted(os.listdir(EMBEDDINGS_DIR)),
    "index": sorted(os.listdir(INDEX_DIR)),
    "metadata": sorted(os.listdir(P4_META_DIR)),
  },
}
with open(os.path.join(P4_META_DIR, "person4_embedding_config.json"), "w") as f:
    json.dump(config, f, indent=4)
print(json.dumps(config, indent=2)[:2500])

{
  "responsible_person": "Person 4",
  "embedding_model": {
    "name": "openai/clip-vit-base-patch32",
    "type": "CLIP ViT-B/32 (image + text encoders)",
    "embedding_dimension": 512,
    "normalisation": "L2",
    "image_batch_size": 64,
    "text_batch_size": 256,
    "text_max_tokens": 77,
    "device": "cuda",
    "library_versions": {
      "torch": "2.11.0+cu130",
      "transformers": "5.22.0",
      "faiss": "1.15.1"
    }
  },
  "image_preprocessing": "EXIF orientation fix, RGB conversion, CLIPImageProcessor (resize+center-crop 224, normalise)",
  "text_preprocessing": "lowercase, strip punctuation, collapse whitespace (indexing only; raw captions preserved)",
  "vector_database": {
    "library": "FAISS",
    "index_type": "IndexIDMap2(IndexFlatIP) \u2014 exact search",
    "similarity_metric": "inner product on L2-normalised vectors (= cosine similarity)",
    "image_index_size": 10000,
    "text_index_size": 50021,
    "image_index_file_mb": 20.56,
    "text_index_fil

In [15]:
# Final deliverables check
expected = {
  EMBEDDINGS_DIR: ["image_embeddings_kb.npy", "image_ids_kb.npy", "text_embeddings_kb.npy",
                   "caption_ids_kb.npy", "caption_image_ids_kb.npy"],
  INDEX_DIR: ["image_index_kb.faiss", "text_index_kb.faiss", "image_id_to_metadata.json", "caption_id_to_text.csv"],
  P4_META_DIR: ["person4_image_metadata.json", "person4_image_metadata.csv", "person4_object_annotations.csv",
                "person4_captions.csv", "person4_test_metadata.json", "person4_embedding_config.json"],
}
ok = True
for folder, files in expected.items():
    for fn in files:
        found = os.path.exists(os.path.join(folder, fn)); ok &= found
        print(("FOUND   " if found else "MISSING ") + fn)
print("\nPERSON 4 DELIVERABLES:", "COMPLETE" if ok else "INCOMPLETE")

FOUND   image_embeddings_kb.npy
FOUND   image_ids_kb.npy
FOUND   text_embeddings_kb.npy
FOUND   caption_ids_kb.npy
FOUND   caption_image_ids_kb.npy
FOUND   image_index_kb.faiss
FOUND   text_index_kb.faiss
FOUND   image_id_to_metadata.json
FOUND   caption_id_to_text.csv
FOUND   person4_image_metadata.json
FOUND   person4_image_metadata.csv
FOUND   person4_object_annotations.csv
FOUND   person4_captions.csv
FOUND   person4_test_metadata.json
FOUND   person4_embedding_config.json

PERSON 4 DELIVERABLES: COMPLETE
